# Pré-treino contrastivo V-LIBRASIL + MALTA → ST-GCN no MINDS

Experimento: pré-treinar com `--ossos --com-z --z-recentrado` e avaliar por LOSO no MINDS. **Não gera entrega (`--final`)**. As referências históricas sem pré-treino são 94,6% e 94,9%; a semente do primeiro controle é `20260916`. O notebook mantém seus hiperparâmetros, mas comparação rigorosa exige controle no mesmo corpus/código/ambiente; os valores históricos não provam pareamento por fold.

## Kaggle: antes de Run All

1. Notebook **privado**, **Internet ligada** (clone; instalação de dependências ausentes) e **Accelerator = GPU**. Usa uma GPU, mesmo em T4 ×2. Não reinstala torch/torchvision do runtime.
2. Importe a versão revisada deste notebook **e publique as correções na branch de código indicada na célula 3**. Atualizar apenas o clone não atualiza as células abertas.
3. Anexe três datasets **privados**. Aceita tar.gz OU pastas já extraídas, em qualquer slug/profundidade:
   - `landmarks-minds.tar.gz` → `landmarks/`: 800 clipes MINDS, 8 pessoas ×20 sinais ×5 repetições, arrays `(T,57,3)`; sidecars opcionais.
   - `landmarks-vlibrasil.tar.gz` → `landmarks-pretreino-auditado/`: corpus auditado, 4.025 clipes na versão local; `.npy` e seus `.npy.proveniencia.json`. O relatório `preparacao.json` é aceito e preservado.
   - `landmarks-malta.tar.gz` → `landmarks-malta/`: corpus completo local, 9.398 clipes com sidecars, incluindo UFSC. Não baixa vídeos nem precisa do PR do downloader UFSC.
4. Se houver mais de uma entrada candidata, configure `ORIGENS` na célula 6. Não há seleção silenciosa da primeira pasta.
5. Input é somente leitura; a cópia validada fica em `/kaggle/working/dados-pretreino-malta`. Os grupos duplicados do pré-treino são **inteiramente excluídos da cópia**, sem escolher rótulo/pessoa; o relatório registra todas as exclusões. A auditoria de isolamento roda depois, sem exceções.

## Retomada e duração

O pré-treino roda 15 épocas e o LOSO roda oito folds de até 120 épocas: reserve várias horas de GPU; a duração real depende do acelerador e corpus. Use o mesmo `NOME_EXPERIMENTO` para repetir células sem criar outra execução. Dados/código/parâmetros e o hash do backbone são conferidos antes de reutilizar resultados.

- Pré-treino completo: reutiliza o backbone. Interrupção antes de salvá-lo: reinicia esse estágio (não há resume do otimizador por época).
- Fine-tuning: reutiliza folds concluídos; o fold interrompido reinicia.
- **Nova sessão Kaggle:** Working não é um backup. Salve outputs privados e anexe-os na nova sessão; indique a pasta do experimento em `RETOMAR_DE` (célula 8). Mantenha o mesmo nome, caminhos de Working, código, dados e parâmetros.

## Licenças e interpretação

MINDS é MIT; V-LIBRASIL tem restrições CC BY-NC-ND; as fontes agregadas pelo MALTA exigem avaliação de permissões, incluindo autorização UFSC pendente. Landmarks não eliminam essas restrições. **Não publique dados, sidecars, outputs ou checkpoints.** O output integral do Kaggle também pode conter a cópia dos landmarks; o pacote de artefatos separado não a inclui.

V03 é reservado para validação do pré-treino, não otimização. MINDS permanece isolado para avaliação LOSO. Identidades agregadas do MALTA não autorizam alegações de LOSO por pessoa nessa fonte. Antes de mudar a decisão de arquitetura, confirme controle e candidato numa semente nova.


## 1. Ambiente e código

In [ ]:
import os, pathlib, shutil, subprocess, sys

EM_KAGGLE = pathlib.Path("/kaggle/working").is_dir() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
EM_COLAB = not EM_KAGGLE and ("google.colab" in sys.modules or pathlib.Path("/content").is_dir())
BASE = pathlib.Path("/kaggle/working" if EM_KAGGLE else "/content" if EM_COLAB else ".").resolve()
URL = "https://github.com/Heitorvazeg/libras-livre-ai-glasses-brasil.git"
BRANCH = "feat/pretreino-gcn-representacao"  # após merge, pode usar dev

def git(*args, repo=None):
    cmd = ["git"] + (["-C", str(repo)] if repo else []) + list(args)
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except subprocess.CalledProcessError as erro:
        raise RuntimeError(f"Git falhou: {erro.stderr.strip()}. No Kaggle, habilite Internet; "
                           "não prosseguir com código desatualizado.") from erro

if EM_KAGGLE or EM_COLAB:
    REPO = BASE / "libras-livre-ai-glasses-brasil"
    if REPO.exists():
        if git("branch", "--show-current", repo=REPO) != BRANCH:
            raise RuntimeError("Clone em outra branch; use outro runtime. Nada será apagado.")
        if git("status", "--porcelain", "--untracked-files=no", repo=REPO):
            raise RuntimeError("Clone tem alterações locais; nada será sobrescrito.")
        git("fetch", "origin", BRANCH, repo=REPO)
        git("merge", "--ff-only", "FETCH_HEAD", repo=REPO)
    else:
        git("clone", "--single-branch", "--branch", BRANCH, URL, str(REPO))
else:
    # Revisão local: usa o workspace existente, sem reset/clone destrutivo.
    REPO = next((p for p in (BASE, *BASE.parents)
                 if (p / "computer-vision-model" / "treino").is_dir()), None)
    if REPO is None:
        raise RuntimeError("Execute localmente dentro do repositório.")

TREINO = (REPO / "computer-vision-model" / "treino").resolve()
if not (TREINO / "entrada_pretreino.py").is_file():
    raise RuntimeError("Este clone ainda não tem a revisão Kaggle. Publique/atualize o PR "
                       "e reimporte também o notebook; atualizar só o clone não troca as células.")
COMMIT_CODIGO = git("rev-parse", "HEAD", repo=REPO)
print("ambiente:", "Kaggle" if EM_KAGGLE else "Colab" if EM_COLAB else "local")
print("código:", COMMIT_CODIGO, "|", TREINO)


In [ ]:
import importlib.util
import tarfile

if sys.version_info < (3, 10) or not hasattr(tarfile, "data_filter"):
    raise RuntimeError("Use Python >=3.10 atualizado, com tarfile.data_filter.")
# Kaggle fornece torch/torchvision/numpy. Não substituir o stack CUDA por pip.
faltantes = [pacote for modulo, pacote in (("yaml", "pyyaml"), ("scipy", "scipy"))
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes], check=True)
try:
    import numpy as np
    import torch, torchvision, yaml, scipy
except Exception as erro:
    raise RuntimeError("Stack torch/torchvision/numpy incompatível. Reinicie com a imagem "
                       "padrão do Kaggle; não reinstale torch isoladamente.") from erro
if not torch.cuda.is_available():
    raise RuntimeError("Ative Settings > Accelerator > GPU; treino completo não roda em CPU.")
THREADS = str(max(1, min(4, os.cpu_count() or 1)))
WORKERS = "2"
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0), "| CUDA:", torch.version.cuda)
print("Usa uma GPU; duas T4 não são somadas automaticamente. Threads:", THREADS)


## 2. Entradas privadas, cópia validada e exclusões auditáveis

**Autoextração do Kaggle:** neste projeto, uploads de `.tar.gz` já apareceram como pastas extraídas em Input. Esta versão aceita ambos, sem exigir reenvio. Se quiser preservar os três `.tar.gz` como arquivos, pode colocá-los na raiz de um `.zip` e enviar esse único dataset privado; confira o conteúdo que apareceu no painel Input. Não é necessário anexar três datasets separados.

Aceita pastas extraídas ou tar.gz. Valida arrays 3D finitos, nomes, fonte, sidecars e hashes antes de instalar. Aceita `.gitkeep` e o relatório de preparação do V-LIBRASIL. Links, arquivos estranhos, entradas ambíguas e corpora incompletos são rejeitados.

Duplicatas por origem/hash de vídeo/hash de landmarks são excluídas **por grupo completo**, apenas na cópia de pré-treino. Input nunca é modificado. `preparacao.json` registra a entrada, os grupos e a saída. Repetir a célula com a mesma entrada reutiliza o destino após conferir os hashes; outra entrada exige outro diretório/runtime.


In [ ]:
import json

# Deixe None para descobrir tar.gz OU pasta extraída, em qualquer slug/profundidade.
# Havendo mais de uma opção, informe o caminho exato em /kaggle/input.
ORIGENS = {"minds": None, "vlibrasil": None, "malta": None}
RAIZ_INPUT = pathlib.Path("/kaggle/input") if EM_KAGGLE else BASE
if EM_COLAB:
    from google.colab import files
    print("Envie os três pacotes privados descritos acima.")
    enviados = files.upload()
    del enviados

sys.path.insert(0, str(TREINO))
import entrada_pretreino as entrada
if pathlib.Path(entrada.__file__).resolve() != TREINO / "entrada_pretreino.py":
    raise RuntimeError("Módulo de outra revisão no kernel: reinicie a sessão.")
origens = {f: entrada.localizar(RAIZ_INPUT, f, ORIGENS[f]) for f in entrada.FONTES}
print("entradas:", {f: str(p) for f, p in origens.items()})

# Dados fora do clone e fora dos artefatos. Nunca escreve em /kaggle/input.
DESTINO = BASE / "dados-pretreino-malta"
PLANO = entrada.preparar(origens, DESTINO)
MINDS = DESTINO / "landmarks"
CORPUS_VLIBRASIL = DESTINO / "landmarks-pretreino-auditado"
CORPUS_MALTA = DESTINO / "landmarks-malta"
print("Relatório de exclusões:", DESTINO / "preparacao.json")
print("Não houve escolha automática de rótulo: TODOS os membros dos grupos duplicados foram excluídos da cópia.")


## 3. Auditoria, teste do ambiente e pré-treino

A lista `REPRESENTACAO` é compartilhada por pré-treino e fine-tuning. O carregamento compara a semântica das flags (incluindo imputação), além dos shapes. A auditoria usa o manifesto versionado do repositório e os hashes do MINDS indicado explicitamente por `--avaliacao`.

São obrigatórios: auditoria, selftest sintético e um forward CUDA curto. Os comandos gravam logs com saída em tempo real. O manifesto do experimento impede misturar código/dados/parâmetros diferentes. Não modifique argumentos depois de executar a célula 8; para outro experimento, altere o nome e reexecute a configuração.


In [ ]:
# Mantenha o nome para retomar; para outro experimento, escolha outro nome.
NOME_EXPERIMENTO = "pretreino-malta-v1"
# Em NOVA sessão: anexe a saída privada anterior e informe a pasta do experimento.
# A estrutura interna deve conter execucao.json, pretreino-... e finetuning-...
RETOMAR_DE = None  # ex.: "/kaggle/input/minha-saida-privada/pretreino-malta-v1"
EXP = BASE / "experimentos-privados" / NOME_EXPERIMENTO
if RETOMAR_DE and not EXP.exists():
    origem_retomada = pathlib.Path(RETOMAR_DE)
    if not (origem_retomada / "execucao.json").is_file():
        raise ValueError("Retomada exige a pasta completa do experimento anterior.")
    shutil.copytree(origem_retomada, EXP)
EXP.mkdir(parents=True, exist_ok=True)
SAIDA_PRE = EXP / "pretreino-vlibrasil-malta-contrastivo"
SAIDA_FT = EXP / "finetuning-minds-loso-com-pretreino"
CHECKPOINT = SAIDA_PRE / "backbone_gcn.pt"
REPRESENTACAO = ["--ossos", "--com-z", "--z-recentrado"]
SEMENTE_BASELINE = "20260916"
PRE_ARGS = [
    "--corpus", str(CORPUS_VLIBRASIL), "--corpus", str(CORPUS_MALTA),
    "--avaliacao", str(MINDS), "--fontes", "vlibrasil,malta",
    "--arquitetura", "gcn", *REPRESENTACAO,
    "--objetivo", "contrastivo", "--pessoa-val", "V03",
    "--epocas", "15", "--lr", "1e-4", "--batch", "64",
    "--p-classes", "32", "--k-exemplos", "2", "--semente", "0",
    "--threads", THREADS, "--workers", WORKERS, "--saida", str(SAIDA_PRE),
]
FT_ARGS = [
    "--arquitetura", "gcn", "--fontes", "minds", *REPRESENTACAO,
    "--landmarks", str(MINDS), "--dispositivo", "cuda",
    "--epocas", "120", "--lr", "1e-3", "--batch", "64",
    "--agendador", "cosseno", "--folds", "0", "--semente", SEMENTE_BASELINE,
    "--threads", THREADS, "--workers", WORKERS,
    "--inicializar", str(CHECKPOINT), "--saida", str(SAIDA_FT),
]

pv = entrada.pv
codigo = {str(p.relative_to(REPO)): pv.hash_arquivo(p)
          for d in (TREINO, TREINO.parent / "datasets", TREINO.parent / "PoC" / "src")
          for p in sorted(d.glob("*.py"))}
config_execucao = {"schema": 1, "entrada_sha256": PLANO["entrada_sha256"],
                  "preparacao_sha256": pv.hash_json(PLANO), "codigo": codigo,
                  "config_sha256": pv.hash_arquivo(TREINO.parent / "PoC" / "config.yaml"),
                  "manifesto_sha256": pv.hash_arquivo(pv.MANIFESTO),
                  "torch": str(torch.__version__), "numpy": str(np.__version__),
                  "pre_args": PRE_ARGS, "ft_args": FT_ARGS}
manifesto_execucao = EXP / "execucao.json"
if manifesto_execucao.exists():
    if json.loads(manifesto_execucao.read_text()) != config_execucao:
        raise RuntimeError("Código, dados, ambiente ou parâmetros mudaram. Use outro NOME_EXPERIMENTO; "
                           "não misture rodadas/checkpoints de execuções diferentes.")
else:
    if any(EXP.iterdir()):
        raise RuntimeError("Experimento contém arquivos sem manifesto; use outro nome.")
    pv.escrever(manifesto_execucao, config_execucao)
shutil.copyfile(DESTINO / "preparacao.json", EXP / "preparacao.json")

def empacotar():
    arquivo = EXP.parent / f"{EXP.name}.tar.gz"
    parcial = arquivo.with_suffix(".parcial")
    with tarfile.open(parcial, "w:gz") as tar:
        tar.add(EXP, arcname=EXP.name)
    parcial.replace(arquivo)
    return arquivo

def executar(script, argumentos, log):
    comando = [sys.executable, "-u", script, *argumentos]
    print("Executando:", " ".join(comando), flush=True)
    with (EXP / log).open("a", encoding="utf-8") as arquivo_log:
        with subprocess.Popen(comando, cwd=TREINO, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as processo:
            for linha in processo.stdout:
                print(linha, end="", flush=True)
                arquivo_log.write(linha)
                arquivo_log.flush()
            if processo.wait():
                raise subprocess.CalledProcessError(processo.returncode, comando)

# Auditoria é obrigatória, incluindo hashes do MINDS montado explicitamente.
executar("pretreinar.py", [*PRE_ARGS, "--dispositivo", "cpu", "--auditar"], "auditoria.log")
executar("selftest.py", [], "selftest.log")
# Smoke CUDA real, sem atualizar pesos, para detectar falhas do stack antes do treino longo.
import gcn
probe = gcn.construir(3, canais_ent=6).cuda().eval()
with torch.no_grad():
    resultado = probe(torch.zeros(2, 6, gcn.T_FIXO, 57, device="cuda"))
    assert resultado.shape == (2, 3) and torch.isfinite(resultado).all()
del probe, resultado
torch.cuda.empty_cache()
print("Auditoria, selftest e smoke CUDA passaram. Saídas privadas:", EXP)


In [ ]:
# Não sobrescrever um backbone que já inicializou rodadas de fine-tuning.
# Pré-treino interrompido ANTES de salvar o backbone reinicia; não há resume por época.
json_checkpoint = CHECKPOINT.with_suffix(".json")
registro_backbone = EXP / "backbone-usado.json"
try:
    if CHECKPOINT.is_file() and json_checkpoint.is_file():
        print("Pré-treino concluído encontrado; reutilizando:", CHECKPOINT)
    else:
        if registro_backbone.exists() or any((SAIDA_FT / "rodadas").glob("*.json")):
            raise RuntimeError("Backbone ausente/incompleto, mas já há fine-tuning. Restaure a saída anterior.")
        executar("pretreinar.py", [*PRE_ARGS, "--dispositivo", "cuda"], "pretreino.log")
    if not CHECKPOINT.is_file() or not json_checkpoint.is_file():
        raise FileNotFoundError("Pré-treino não produziu backbone e metadados.")
    # Apenas checkpoints privados produzidos por esta execução; torch.load usa pickle.
    import treinar
    from argparse import Namespace
    modelo_verificacao = gcn.construir(3, canais_ent=6)
    treinar.aplicar_backbone(modelo_verificacao, CHECKPOINT, "gcn", Namespace(
        com_z=True, z_recentrado=True, ossos=True, movimento=False, sem_imputacao=False))
    del modelo_verificacao
    identidade = {"checkpoint_sha256": pv.hash_arquivo(CHECKPOINT),
                  "metadados_sha256": pv.hash_arquivo(json_checkpoint)}
    if registro_backbone.exists() and json.loads(registro_backbone.read_text()) != identidade:
        raise RuntimeError("Backbone mudou no mesmo experimento; não misture rodadas.")
    pv.escrever(registro_backbone, identidade)
    print("Backbone validado:", identidade["checkpoint_sha256"])
finally:
    print("Backup privado:", empacotar())


## 4. Fine-tuning LOSO no MINDS

Mantém semente `20260916`, representação e hiperparâmetros do controle histórico, adicionando `--inicializar`. `--folds 0` executa os oito folds. O diretório MINDS é passado explicitamente; nenhum dado é buscado no clone.

O mesmo experimento reutiliza os JSONs de folds concluídos; o hash do backbone impede trocar a inicialização no meio. Para uma nova sessão, restaure o diretório completo via `RETOMAR_DE` antes de executar a configuração. **Não há `--final`.**


In [ ]:
# FT_ARGS é definido junto de PRE_ARGS e registrado em execucao.json.
if not CHECKPOINT.is_file() or not registro_backbone.is_file():
    raise FileNotFoundError("Execute a validação do backbone antes do fine-tuning.")
if json.loads(registro_backbone.read_text())["checkpoint_sha256"] != pv.hash_arquivo(CHECKPOINT):
    raise RuntimeError("Backbone alterado desde a validação; fine-tuning recusado.")
try:
    executar("treinar.py", FT_ARGS, "finetuning.log")
finally:
    # Também preserva rodadas concluídas se o subprocesso falhar.
    # Encerramento forçado do runtime exige salvar a saída privada do Kaggle.
    print("Backup privado (inclusive parciais):", empacotar())


## 5. O número, contra o baseline conhecido

In [ ]:
# JSONs das rodadas são a fonte numérica; não depender de regex no Markdown arredondado.
registros = [json.loads(p.read_text(encoding="utf-8"))
             for p in sorted((SAIDA_FT / "rodadas").glob("*.json"))]
folds = {r["teste"]: r["acuracia"] * 100 for r in registros}
pessoas_esperadas = {pv.identidade_nome(p)[0] for p in MINDS.glob("*.npy")}
if len(registros) != 8 or set(folds) != pessoas_esperadas:
    raise RuntimeError("LOSO incompleto: não comparar uma média parcial com o baseline de oito pessoas.")
if any(not np.isfinite(v) or not 0 <= v <= 100 for v in folds.values()):
    raise ValueError("Acurácia inválida nas rodadas.")
media = float(np.mean(list(folds.values())))
print(f"{'rodada':<10}{'acurácia':>10}")
for pessoa, acc in sorted(folds.items()):
    print(f"{pessoa:<10}{acc:>9.1f}%")
print(f"\nMÉDIA com pré-treino: {media:.2f}%")
print(f"Referência histórica, semente 20260916: 94.6% | diferença {media - 94.6:+.2f} pp")
print("Referência histórica de outra semente: 94.9% (não é comparação pareada).")
print("""
INTERPRETAÇÃO:
  - ~1,7 pp de variação histórica não é um limiar de significância: uma diferença
    pequena, sozinha, não permite separar efeito de ruído.
  - Os 94.6% são referência histórica arredondada; mesmos hiperparâmetros/semente
    não garantem igualdade entre versões de código, CUDA, dados e hardware.
  - Para medir consistência por fold, é necessário obter as rodadas do controle
    ou rerodar o controle sem --inicializar no mesmo ambiente e corpus.
  - Antes de decidir, confirmar CONTROLE e CANDIDATO numa semente nova.
""")


## 6. Baixar todos os artefatos — uso privado

Inclui backbone `.pt`, JSON de metadados, relatório e matriz de confusão de
cada rodada. Landmarks e pacotes de entrada ficam fora.

In [ ]:
for rel in sorted(EXP.rglob("relatorio.md")):
    print("=" * 70, "\n", rel.relative_to(EXP))
    print(rel.read_text(encoding="utf-8")[:1500])
arquivo = empacotar()
print("Arquivo privado:", arquivo, "|", arquivo.stat().st_size, "bytes")
print("Inclui manifestos, exclusões, backbone, metadados, logs e rodadas; não inclui landmarks de entrada.")
if EM_COLAB:
    from google.colab import files
    files.download(str(arquivo))
else:
    from IPython.display import FileLink, display
    display(FileLink(str(arquivo.relative_to(BASE))))
    print("Kaggle: Save Version e mantenha notebook, datasets e outputs PRIVADOS.")
    print("Uma nova sessão não preserva Working automaticamente: anexe a saída privada anterior e configure RETOMAR_DE.")
    print("O painel Output pode incluir dados-pretreino-malta além deste arquivo; não publique o output completo.")
